# EcoSignal GeoRisk V2 — 1 m tiled LiDAR analysis and model export

**Run cells in order in Google Colab.** This version replaces the earlier coarse-grid, all-in-memory V2 notebook. It expects the **already-aligned** `aligned_dems_1m/Dem_{year}_aligned.tif` rasters; do not reproject them again. Their grid should be the original 2015 USGS EPSG:32119 1 m reference (38,511 × 39,068 or similar). The raw 2024 US survey-foot elevations must already be converted to **meters** during alignment.

**Stages:** dependencies → paths and QA → tile identification → 1 m terrain features → 2015–2022 conservative labels and spatially separate train/dev sampling → RF/HGB/XGBoost selection → **only then** 2022–2024 temporal + geographic holdout evaluation → optional 1 m bias diagnostics → cropped 2024 susceptibility GeoTIFF → lightweight WGS84 PNG overlay and integration manifest.

**Important science limitations**
- **No automatic vertical corrections.** Prior 26 m candidate-terrain diagnostics found 2022–2024 median +0.021 m and block variation 0.0503 m; these do **not** justify applying an offset. 2013 and 2022 geoid versions are still assumed, not verified; 2024 GEOID18 is documented.
- 2015–2022 USGS propagated vertical error was ~0.0512 m, but the **2022–2024 combined error has not been verified**. Default label thresholds (0.105 m, 0.15 m) are **provisional conservative operating choices**, not certified survey-level detection limits.
- **Different observation periods:** training labels represent detectable lowering between 2015 and 2022 (about seven years); testing labels reflect 2022 to 2024 (about two years). Do not interpret a successful test as a calibrated probability or fixed-horizon forecast.
- **Terrain-only model:** the feature set below comprises local slope, topographic position, curvature proxies, roughness and multiscale relief. **True flow accumulation / TWI / drainage-network distances are intentionally omitted**, because computing those independently in tiles creates erroneous drainage boundaries. Add them later only if generated consistently from a full catchment-aware hydrologic mosaic.
- An erosion label here means **observed terrain lowering above an assumed threshold**, not independently field-confirmed erosion; construction, water bodies, vegetation artifacts and LiDAR processing can cause false labels. External validation is still required.
- The temporal holdout is evaluated **once** after the model/threshold are frozen. If you retune based on it, it is no longer an untouched final test.

**Output location:** `/content/drive/MyDrive/EcoSignal_DEM/geomorphology_v2_1m/`. Original files and your V1 backup are untouched. The produced PNG is an **illustrative web-map overlay**. The GeoTIFF is the high-resolution authoritative map artifact.
**Reviewed execution copy:** square 63 m measured-data mask, corrected northward grade sign, versioned sample-cache invalidation, frozen-model checkpoint before holdout, and holdout statistics saved before map export. This copy has not been executed on the regional DEMs.


In [ ]:
# CELL 1 — Install packages / mount Google Drive (Colab)
%pip -q install -U rasterio scipy scikit-learn xgboost pandas joblib matplotlib pillow
from google.colab import drive
drive.mount('/content/drive')
print('Drive and packages ready.')

In [ ]:
# CELL 2 — Configuration. Change ONLY these values if your files differ.
from pathlib import Path
import os, json, hashlib, shutil, math
import numpy as np
import rasterio

BASE = Path('/content/drive/MyDrive/EcoSignal_DEM')
INPUT = Path('/content/aligned_dems_1m')
# Set ECOSIGNAL_OUTPUT_DIR to a runtime-disk directory if Drive is full.
# Runtime-disk outputs must be downloaded before the Colab session ends.
OUT = Path(os.environ.get('ECOSIGNAL_OUTPUT_DIR', str(BASE / 'geomorphology_v2_1m')))
FEATURE_IMPLEMENTATION_VERSION = 'square-footprint-north-grade-v2'
OUT.mkdir(parents=True, exist_ok=True)

DEM_FILES = {y: INPUT / f'Dem_{y}_aligned.tif' for y in [2013, 2015, 2022, 2024]}
YEARS_USED_FOR_MODEL = (2015, 2022, 2024)

# Runtime configuration. Each 512x512 tile is processed with a 48-pixel halo.
TILE = 512
HALO = 48
BLOCK_M = 2000                   # contiguous spatial splitting units
BUFFER_M = 200                   # exclusion from each block boundary
SEED = 20261011

# Pre-declared, conservative screening thresholds (in METERS).
# Re-evaluate after survey vertical references and field/control data are verified.
LOWERING_THRESHOLD_2015_2022_M = 0.105
LOWERING_THRESHOLD_2022_2024_M = 0.150
LABEL_AMBIGUITY_BAND_M = 0.030
MAX_ABSOLUTE_CHANGE_M = 4.0      # exclude huge differences that need separate QA

TRAIN_SAMPLES_PER_CLASS = 25000  # balanced training set: up to 50,000 total
DEV_SAMPLES = 60000              # UNBALANCED random samples (natural prevalence)
TEST_SAMPLES = 80000             # UNBALANCED random samples, held out

REUSE_SAMPLE_CACHE = True        # resumes after disconnect if config unchanged
# Reading directly from Drive may be slow. Enable if local disk fits all three files.
STAGE_INPUTS_TO_COLAB_DISK = False

for y in YEARS_USED_FOR_MODEL:
    if not DEM_FILES[y].exists():
        raise FileNotFoundError(f'Missing {y} aligned 1m DEM: {DEM_FILES[y]}')

if STAGE_INPUTS_TO_COLAB_DISK:
    local = Path('/content/georisk1m_inputs')
    local.mkdir(exist_ok=True)
    need = sum(DEM_FILES[y].stat().st_size for y in YEARS_USED_FOR_MODEL)
    free = shutil.disk_usage('/content').free
    assert free > need * 1.20, f'Not enough temporary disk: need ~{need/1e9:.1f} GB, free {free/1e9:.1f} GB'
    for y in YEARS_USED_FOR_MODEL:
        p = local / DEM_FILES[y].name
        if not p.exists() or p.stat().st_size != DEM_FILES[y].stat().st_size:
            print('Copying to fast local Colab disk:', y)
            shutil.copy2(DEM_FILES[y], p)
        DEM_FILES[y] = p

with rasterio.open(DEM_FILES[2015]) as ref:
    GRID_CRS = ref.crs
    GRID_TRANSFORM = ref.transform
    GRID_SHAPE = ref.shape
    H, W = ref.shape
    GRID_PROFILE = ref.profile.copy()
    assert GRID_CRS.to_epsg() == 32119, f'Unexpected reference CRS {GRID_CRS}'
    assert np.isclose(ref.res[0], 1.0) and np.isclose(ref.res[1], 1.0)

for y in YEARS_USED_FOR_MODEL:
    with rasterio.open(DEM_FILES[y]) as ds:
        assert ds.crs == GRID_CRS, f'{y}: CRS differs!'
        assert ds.shape == GRID_SHAPE, f'{y}: shape differs!'
        assert ds.transform.almost_equals(GRID_TRANSFORM), f'{y}: transform differs!'
        assert ds.nodata is not None, f'{y}: missing nodata declaration'
        print(y, 'shape:', ds.shape, 'resolution:', ds.res,
              'GB:', round(Path(DEM_FILES[y]).stat().st_size / 1e9, 3),
              'Z metadata:', ds.tags().get('vertical_units', 'verify externally'))

if DEM_FILES[2013].exists():
    print('2013 source retained (not used in V2 model training).')
print('Output:', OUT)
print('Full reference grid:', f'{W:,} x {H:,}', f'({W*H:,} pixels)')
print('No vertical offsets will be applied.')

In [ ]:
# CELL 3 — Tile utilities and truthful 1 m local terrain features.
# These 21 features DO NOT claim true catchment hydrology.
from scipy.ndimage import (uniform_filter, distance_transform_edt,
                           binary_erosion)
from rasterio.windows import Window, transform as window_transform

FEATURE_NAMES = [
    'elevation_m', 'slope_deg', 'slope_percent',
    'grade_east', 'grade_north', 'aspect_east', 'aspect_north',
    'curvature_laplacian_proxy', 'profile_curvature_proxy',
    'plan_curvature_proxy',
    'tpi_5m', 'tpi_15m', 'tpi_61m',
    'roughness_5m', 'roughness_15m', 'roughness_61m',
    'smoothed_slope_15m', 'smoothed_slope_61m',
    'tri_3m_proxy', 'valley_depth_61m', 'ridge_height_61m'
]

def read_core(ds, win):
    a = ds.read(1, window=win, masked=True)
    return a.filled(np.nan).astype('float32')

def read_with_halo(ds, win):
    expanded = Window(win.col_off - HALO, win.row_off - HALO,
                      win.width + 2*HALO, win.height + 2*HALO)
    a = ds.read(1, window=expanded, boundless=True, masked=True)
    return a.filled(np.nan).astype('float32')

def valid_feature_mask(raw_with_halo, win):
    good = np.isfinite(raw_with_halo)
    # Require valid measured elevations throughout the local 61 m feature
    # footprint to avoid inventing features across survey/data boundaries.
    # The 61x61 mean plus its gradient uses a square 63x63 footprint.
    # A Euclidean distance margin misses invalid pixels near square corners.
    # minimum_filter is linear-time per axis and avoids an expensive dense erosion.
    from scipy.ndimage import minimum_filter
    measured_footprint = minimum_filter(good.astype(np.uint8), size=63,
                                       mode='constant', cval=0).astype(bool)
    center = (slice(HALO, HALO + int(win.height)),
              slice(HALO, HALO + int(win.width)))
    return good[center] & measured_footprint[center]

def build_terrain_features(raw_with_halo, win):
    finite = np.isfinite(raw_with_halo)
    if not finite.any():
        h,w = int(win.height),int(win.width)
        return np.full((h,w,len(FEATURE_NAMES)), np.nan, dtype=np.float32)

    # Nearest filling is used *only* for convolution. Pixels near data edges
    # are later EXCLUDED by valid_feature_mask (32 pixel margin).
    _, indices = distance_transform_edt(~finite,
                        return_distances=True, return_indices=True)
    z = raw_with_halo[tuple(indices)].astype(np.float32)
    mean5  = uniform_filter(z, size=5, mode='nearest')
    mean15 = uniform_filter(z, size=15, mode='nearest')
    mean61 = uniform_filter(z, size=61, mode='nearest')
    sq = z*z
    rough5  = np.sqrt(np.maximum(uniform_filter(sq,5)-mean5**2, 0))
    rough15 = np.sqrt(np.maximum(uniform_filter(sq,15)-mean15**2,0))
    rough61 = np.sqrt(np.maximum(uniform_filter(sq,61)-mean61**2,0))

    gy,gx = np.gradient(z, 1.0, 1.0)
    slope = np.hypot(gx,gy)
    slope_deg = np.degrees(np.arctan(slope))
    # Aspect sin/cos: eastward/northward downhill exposure proxies.
    east = -gx/np.maximum(slope, 1.e-5)
    north = gy/np.maximum(slope, 1.e-5)

    gyy,gyx = np.gradient(gy, 1., 1.)
    gxy,gxx = np.gradient(gx, 1., 1.)
    lap = gxx + gyy
    denominator = np.maximum(gx*gx + gy*gy, 1.e-6)
    profile = (gxx*gx*gx + (gxy+gyx)*gx*gy + gyy*gy*gy) / denominator
    plan = lap - profile
    sm15y,sm15x = np.gradient(mean15, 1., 1.)
    sm61y,sm61x = np.gradient(mean61, 1., 1.)
    mean3 = uniform_filter(z,3)
    tri3 = np.sqrt(np.maximum(uniform_filter(sq,3)-2*z*mean3+z*z,0))
    tpi61 = z - mean61

    # Raster rows increase southward; northward elevation grade is -gy.
    features = [z,slope_deg,slope*100.,gx,-gy,east,north,lap,profile,plan,
                z-mean5,z-mean15,tpi61,rough5,rough15,rough61,
                np.degrees(np.arctan(np.hypot(sm15x,sm15y))),
                np.degrees(np.arctan(np.hypot(sm61x,sm61y))),
                tri3,np.maximum(-tpi61,0),np.maximum(tpi61,0)]
    center = (slice(HALO, HALO + int(win.height)),
              slice(HALO, HALO + int(win.width)))
    cube = np.stack([a[center] for a in features], axis=-1).astype('float32')
    assert cube.shape[-1] == len(FEATURE_NAMES)
    return cube

def label_lowering(earlier, later, cutoff_m):
    # 1 = clear terrain lowering; 0 = no detectable lowering;
    # -1 = ambiguous / invalid. No median bias subtraction.
    dz = later.astype('float32') - earlier.astype('float32')
    valid = np.isfinite(dz) & (np.abs(dz) <= MAX_ABSOLUTE_CHANGE_M)
    y = np.full(dz.shape, -1, dtype=np.int8)
    y[valid & (dz <= -cutoff_m)] = 1
    y[valid & (dz >= -(cutoff_m - LABEL_AMBIGUITY_BAND_M))] = 0
    return y

def spatial_partition(win):
    # Fixed 2 km blocks: 65% train / 20% dev / 15% temporal+spatial test.
    # A 200 m interior guard removes samples close to ANY block boundary.
    rr = np.arange(int(win.row_off), int(win.row_off + win.height), dtype=np.int64)[:,None]
    cc = np.arange(int(win.col_off), int(win.col_off + win.width), dtype=np.int64)[None,:]
    bx,by = cc//BLOCK_M, rr//BLOCK_M
    hashed = ((bx * 73856093) ^ (by * 19349663) ^ SEED) % 100
    guard_x = (cc % BLOCK_M >= BUFFER_M) & (cc % BLOCK_M < BLOCK_M-BUFFER_M)
    guard_y = (rr % BLOCK_M >= BUFFER_M) & (rr % BLOCK_M < BLOCK_M-BUFFER_M)
    safe = guard_x & guard_y
    return np.where(hashed < 65,0,np.where(hashed < 85,1,2)).astype(np.uint8),safe

print('Defined',len(FEATURE_NAMES),'tile-safe terrain features.')

In [ ]:
# CELL 4 — Identify only 1 m tiles with overlapping 2015, 2022 and 2024 coverage.
# This reads 2024 coverage but NOT observed 2022–2024 elevation differences.
import time
TILES = []
coverage_pixels = 0
started = time.time()

with rasterio.open(DEM_FILES[2015]) as d15, \
     rasterio.open(DEM_FILES[2022]) as d22, \
     rasterio.open(DEM_FILES[2024]) as d24:
    tile_counter = 0
    for r0 in range(0,H,TILE):
        for c0 in range(0,W,TILE):
            win = Window(c0,r0,min(TILE,W-c0),min(TILE,H-r0))
            a24 = read_core(d24,win)
            if np.isfinite(a24).sum() < 100:
                continue
            a15 = read_core(d15,win)
            a22 = read_core(d22,win)
            n = int(np.count_nonzero(np.isfinite(a24)&np.isfinite(a15)&np.isfinite(a22)))
            if n < 100:
                continue
            TILES.append((int(r0),int(c0),int(win.height),int(win.width)))
            coverage_pixels += n
        tile_counter += 1
        if tile_counter % 10 == 0:
            print(f'Checked {tile_counter}/{math.ceil(H/TILE)} tile rows; retained {len(TILES)} tiles',end='\r')

assert len(TILES) >= 10, 'Too little common coverage: inspect aligned_dems_1m and NoData handling.'
print(f'\nOverlap: {len(TILES)} tiles; approximately {coverage_pixels:,} valid 1 m pixels')
print('Elapsed minutes:',round((time.time()-started)/60,1))

# Bounds are tile-aligned, not a promise of complete valid coverage inside.
MIN_R = min(t[0] for t in TILES)
MIN_C = min(t[1] for t in TILES)
MAX_R = max(t[0]+t[2] for t in TILES)
MAX_C = max(t[1]+t[3] for t in TILES)
print('Output raster crop: rows',MIN_R,MAX_R,'cols',MIN_C,MAX_C)
(OUT/'tile_manifest.json').write_text(json.dumps({
    'tile_size':TILE,'reference_grid_shape':[H,W],
    'overlap_tiles':[list(t) for t in TILES],
    'approximate_overlap_pixels':coverage_pixels,
    'crop_rows':[MIN_R,MAX_R],'crop_cols':[MIN_C,MAX_C]
}))

In [ ]:
# CELL 5 — Bounded uniform-priority random sampling.
# Dev/test are sampled WITHOUT class balancing, so prevalence-based metrics
# are meaningful for the eligible pixels in the observed overlap footprint.
import joblib

class PrioritySampler:
    def __init__(self, capacity, seed):
        self.capacity = int(capacity)
        self.rng = np.random.default_rng(seed)
        self.keys = np.empty(0,dtype=np.float64)
        self.X = np.empty((0,len(FEATURE_NAMES)),dtype=np.float32)
        self.y = np.empty(0,dtype=np.int8)

    def add(self, cube, labels, mask):
        idx = np.flatnonzero(mask.ravel())
        n = len(idx)
        if not n:return
        keys = self.rng.random(n)
        if len(self.keys) == self.capacity:
            keep = keys < self.keys.max()
            idx,keys = idx[keep],keys[keep]
        if not len(idx):return
        if len(keys)>self.capacity:
            chosen = np.argpartition(keys,self.capacity-1)[:self.capacity]
            idx,keys = idx[chosen],keys[chosen]
        xx = cube.reshape(-1,len(FEATURE_NAMES))[idx]
        yy = labels.ravel()[idx]
        self.X = np.concatenate((self.X,xx),axis=0)
        self.y = np.concatenate((self.y,yy),axis=0)
        self.keys = np.concatenate((self.keys,keys),axis=0)
        if len(self.keys)>self.capacity:
            chosen=np.argpartition(self.keys,self.capacity-1)[:self.capacity]
            self.keys=self.keys[chosen]
            self.X=self.X[chosen]
            self.y=self.y[chosen]

    def arrays(self):
        return self.X, self.y

# Changes to files, thresholds, feature order or split invalidate the cache.
source_key=[]
for year in YEARS_USED_FOR_MODEL:
    p=DEM_FILES[year]
    source_key.append([year,p.name,p.stat().st_size,int(p.stat().st_mtime)])
signature_input = {
    'feature_implementation_version':FEATURE_IMPLEMENTATION_VERSION,
    'sources':source_key,'tile':TILE,'halo':HALO,'seed':SEED,
    'block':BLOCK_M,'buffer':BUFFER_M,
    'threshold15_22':LOWERING_THRESHOLD_2015_2022_M,
    'ambiguity':LABEL_AMBIGUITY_BAND_M,
    'max_abs':MAX_ABSOLUTE_CHANGE_M,
    'train_cap':TRAIN_SAMPLES_PER_CLASS,'dev_cap':DEV_SAMPLES,
    'features':FEATURE_NAMES,'tile_coords':TILES
}
CACHE_SIGNATURE=hashlib.sha256(json.dumps(signature_input,sort_keys=True).encode()).hexdigest()
TRAIN_CACHE=OUT/'pre2024_train_dev_samples.joblib'
print('Cache signature:',CACHE_SIGNATURE[:12])

In [ ]:
# CELL 6 — Build PRE-2024 labels from 2015->2022, extract features from 2015 ONLY.
# 2024 is used only to restrict to common coverage; never for training labels.
if REUSE_SAMPLE_CACHE and TRAIN_CACHE.exists():
    bundle=joblib.load(TRAIN_CACHE)
    if bundle.get('signature') == CACHE_SIGNATURE:
        X_train,y_train,X_dev,y_dev = (bundle[k] for k in ['X_train','y_train','X_dev','y_dev'])
        pre2024_counts = bundle['counts']
        print('Loaded verified cached pre-2024 train/dev samples.')
    else:
        bundle = None
else:
    bundle = None

if bundle is None:
    train_pos=PrioritySampler(TRAIN_SAMPLES_PER_CLASS,SEED+1)
    train_neg=PrioritySampler(TRAIN_SAMPLES_PER_CLASS,SEED+2)
    dev_all=PrioritySampler(DEV_SAMPLES,SEED+3)
    pre2024_counts={'train_0':0,'train_1':0,'dev_0':0,'dev_1':0,
                    'ambiguous_or_invalid':0}

    with rasterio.open(DEM_FILES[2015]) as d15, \
         rasterio.open(DEM_FILES[2022]) as d22, \
         rasterio.open(DEM_FILES[2024]) as d24:
        for i,(r,c,h,w) in enumerate(TILES):
            win=Window(c,r,w,h)
            a15=read_core(d15,win)
            a22=read_core(d22,win)
            a24_valid=np.isfinite(read_core(d24,win))  # COVERAGE ONLY
            cube_input=read_with_halo(d15,win)
            local_features=build_terrain_features(cube_input,win)
            local_valid=valid_feature_mask(cube_input,win)
            region=(np.isfinite(a15)&np.isfinite(a22)&a24_valid&local_valid&
                    np.all(np.isfinite(local_features),axis=2))
            labels=label_lowering(a15,a22,LOWERING_THRESHOLD_2015_2022_M)
            partitions,safe=spatial_partition(win)
            region &= safe
            for val,split in [(0,'train'),(1,'dev')]:
                m=region&(partitions==val)
                pre2024_counts[f'{split}_0']+=int(np.count_nonzero(m&(labels==0)))
                pre2024_counts[f'{split}_1']+=int(np.count_nonzero(m&(labels==1)))
            pre2024_counts['ambiguous_or_invalid']+=int(np.count_nonzero(region&(labels<0)))
            train_pos.add(local_features,labels,region&(partitions==0)&(labels==1))
            train_neg.add(local_features,labels,region&(partitions==0)&(labels==0))
            dev_all.add(local_features,labels,region&(partitions==1)&(labels>=0))
            if (i+1)%25==0 or i+1==len(TILES):
                print(f'Training samples: tile {i+1}/{len(TILES)}')

    Xp,yp=train_pos.arrays()
    Xn,yn=train_neg.arrays()
    X_train=np.concatenate((Xp,Xn),axis=0)
    y_train=np.concatenate((yp,yn),axis=0)
    X_dev,y_dev=dev_all.arrays()
    rng=np.random.default_rng(SEED)
    permutation=rng.permutation(len(y_train))
    X_train,y_train=X_train[permutation],y_train[permutation]
    joblib.dump({'signature':CACHE_SIGNATURE,'X_train':X_train,
                 'y_train':y_train,'X_dev':X_dev,'y_dev':y_dev,
                 'counts':pre2024_counts},TRAIN_CACHE,compress=3)
    print('Saved pre-2024 sample checkpoint:',TRAIN_CACHE)

print('Training sample shape:',X_train.shape,'positive count:',int(y_train.sum()))
print('Development sample shape:',X_dev.shape,'positive count:',int(y_dev.sum()))
print('Underlying eligible source counts:',pre2024_counts)
assert len(np.unique(y_train))==2 and len(np.unique(y_dev))==2, (
    'Insufficient training/development erosion labels. Investigate overlap and label thresholds; do not invent positive labels.')

In [ ]:
# CELL 7 — Train multiple algorithms; select only on spatially held-out 2015->2022 DEV.
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score,average_precision_score,
            precision_recall_curve,precision_score,recall_score,f1_score,
            confusion_matrix,brier_score_loss)
from sklearn.base import clone
from xgboost import XGBClassifier
import pandas as pd
from IPython.display import display

MODELS={
    'RandomForest':RandomForestClassifier(
         n_estimators=200,max_depth=18,min_samples_leaf=6,
         max_features='sqrt',random_state=SEED,n_jobs=-1),
    'HistGradientBoosting':HistGradientBoostingClassifier(
         max_iter=175,learning_rate=0.06,max_leaf_nodes=31,
         min_samples_leaf=25,l2_regularization=1.0,random_state=SEED),
    'XGBoost':XGBClassifier(
         n_estimators=250,max_depth=5,learning_rate=0.055,
         subsample=0.85,colsample_bytree=0.85,reg_lambda=3.0,
         tree_method='hist',objective='binary:logistic',
         eval_metric='logloss',n_jobs=2,random_state=SEED)
}

def score_metrics(y,p,threshold):
    y=np.asarray(y);p=np.asarray(p)
    if len(np.unique(y))<2:
        raise ValueError('Cannot compute ROC-AUC/PR-AUC with only one class.')
    hit=p >= threshold
    top=p >= np.quantile(p,0.90)
    baseline=float(np.mean(y))
    return {
        'roc_auc':float(roc_auc_score(y,p)),
        'pr_auc':float(average_precision_score(y,p)),
        'precision':float(precision_score(y,hit,zero_division=0)),
        'recall':float(recall_score(y,hit,zero_division=0)),
        'f1':float(f1_score(y,hit,zero_division=0)),
        'brier_uncalibrated':float(brier_score_loss(y,p)),
        'eligible_sample_prevalence':baseline,
        'top_decile_lift':float(y[top].mean()/baseline) if top.any() and baseline>0 else float('nan'),
        'top_decile_recall':float(y[top].sum()/max(1,y.sum())),
        'n':int(len(y))}

def choose_dev_threshold(y,p):
    precisions,recalls,thresholds=precision_recall_curve(y,p)
    f1s=2*precisions[:-1]*recalls[:-1]/np.maximum(precisions[:-1]+recalls[:-1],1e-12)
    return float(thresholds[int(np.argmax(f1s))])

DEV_RESULTS={}
trained={}
for name,model in MODELS.items():
    print('Fitting',name)
    model.fit(X_train,y_train)
    p=model.predict_proba(X_dev)[:,1]
    threshold=choose_dev_threshold(y_dev,p)
    metrics=score_metrics(y_dev,p,threshold)
    DEV_RESULTS[name]={**metrics,'selected_threshold':threshold}
    trained[name]=model
    print(name,'DEV PR-AUC:',round(metrics['pr_auc'],4),
          'ROC-AUC:',round(metrics['roc_auc'],4),
          'threshold:',round(threshold,4))

display(pd.DataFrame.from_dict(DEV_RESULTS,orient='index').round(4))
WINNER=max(DEV_RESULTS, key=lambda name:(DEV_RESULTS[name]['pr_auc'],DEV_RESULTS[name]['roc_auc']))
FINAL_MODEL=trained[WINNER]
FROZEN_THRESHOLD=DEV_RESULTS[WINNER]['selected_threshold']
print('MODEL FROZEN:',WINNER,'threshold selected on 2015–2022 DEV:',FROZEN_THRESHOLD)
print('2022–2024 observed differences have NOT been used for training or model selection.')

# Persist the exact frozen model BEFORE independent evaluation or full-map export.
FROZEN_MODEL_FILE = OUT / 'ecosignal_georisk_v2_frozen_checkpoint.joblib'
frozen_bundle = {
    'model': FINAL_MODEL, 'winner': WINNER, 'threshold': FROZEN_THRESHOLD,
    'development_results': DEV_RESULTS, 'sample_signature': CACHE_SIGNATURE,
    'feature_order': FEATURE_NAMES,
    'feature_implementation_version': FEATURE_IMPLEMENTATION_VERSION,
}
frozen_temp = FROZEN_MODEL_FILE.with_suffix('.pending.joblib')
joblib.dump(frozen_bundle, frozen_temp, compress=3)
os.replace(frozen_temp, FROZEN_MODEL_FILE)
print('Frozen model checkpoint saved:', FROZEN_MODEL_FILE)


In [ ]:
# CELL 8 — ONE-TIME external test: 2022 terrain -> 2022–2024 observed lowering.
# Uses only 15% geographic holdout spatial blocks. Do not tune after reading this.
test_sample=PrioritySampler(TEST_SAMPLES,SEED+4)
test_counts={'eligible_0':0,'eligible_1':0,'ambiguous_or_invalid':0}

with rasterio.open(DEM_FILES[2015]) as d15, \
     rasterio.open(DEM_FILES[2022]) as d22, \
     rasterio.open(DEM_FILES[2024]) as d24:
    for i,(r,c,h,w) in enumerate(TILES):
        win=Window(c,r,w,h)
        a15_valid=np.isfinite(read_core(d15,win))
        a22=read_core(d22,win)
        a24=read_core(d24,win)
        input22=read_with_halo(d22,win)
        features22=build_terrain_features(input22,win)
        valid22=valid_feature_mask(input22,win)
        parts,safe=spatial_partition(win)
        eligible=(a15_valid&np.isfinite(a22)&np.isfinite(a24)&valid22&
                 safe&(parts==2)&np.isfinite(features22).all(axis=2))
        labels24=label_lowering(a22,a24,LOWERING_THRESHOLD_2022_2024_M)
        test_counts['eligible_0']+=int(np.count_nonzero(eligible&(labels24==0)))
        test_counts['eligible_1']+=int(np.count_nonzero(eligible&(labels24==1)))
        test_counts['ambiguous_or_invalid']+=int(np.count_nonzero(eligible&(labels24<0)))
        test_sample.add(features22,labels24,eligible&(labels24>=0))
        if (i+1)%25==0 or i+1==len(TILES):
            print(f'Temporal test sampling: tile {i+1}/{len(TILES)}')

X_test,y_test=test_sample.arrays()
print('Unbalanced spatial+temporal test sample:',X_test.shape,'positives:',int(y_test.sum()))
print('Test source counts:',test_counts)
assert len(np.unique(y_test))==2, (
    'External test has fewer than two classes. No ROC/PR metrics can be claimed; review coverage or predeclared thresholds scientifically.')
prob_test=FINAL_MODEL.predict_proba(X_test)[:,1]
FINAL_METRICS=score_metrics(y_test,prob_test,FROZEN_THRESHOLD)
FINAL_METRICS.update({'model':WINNER,'threshold':FROZEN_THRESHOLD,
                      'test':'2022-2024 independent survey interval and geographically held-out blocks',
                      'lowering_threshold_m':LOWERING_THRESHOLD_2022_2024_M})
display(pd.DataFrame([FINAL_METRICS]).round(4))
print('Confusion matrix at FROZEN development threshold:')
print(confusion_matrix(y_test,prob_test>=FROZEN_THRESHOLD))
print('Original GeoRisk V1 ROC-AUC=0.7624 was measured on a DIFFERENT evaluation design.')
print('Do not claim an apples-to-apples improvement.')

# Preserve holdout evidence immediately; map export can be resumed separately.
EVALUATION_FILE = OUT / 'ecosignal_georisk_v2_holdout_checkpoint.json'
evaluation = {
    'sample_signature': CACHE_SIGNATURE,
    'feature_implementation_version': FEATURE_IMPLEMENTATION_VERSION,
    'metrics': FINAL_METRICS, 'eligible_source_counts': test_counts,
    'confusion_matrix': confusion_matrix(y_test, prob_test >= FROZEN_THRESHOLD).tolist(),
    'frozen_model_file': FROZEN_MODEL_FILE.name,
    'warning': 'Do not tune the model or thresholds using these holdout results.',
}
evaluation_temp = EVALUATION_FILE.with_suffix('.pending.json')
evaluation_temp.write_text(json.dumps(evaluation, indent=2), encoding='utf-8')
os.replace(evaluation_temp, EVALUATION_FILE)
print('Independent evaluation checkpoint saved:', EVALUATION_FILE)


In [ ]:
# CELL 9 — OPTIONAL 1 m candidate stable-ground offsets (diagnostic only).
# Run after the final test; does NOT change model or bias-correct any rasters.
# Terrain-based candidates are NOT independent benchmarks or verified unchanged roads.
STABLE_QA_ROWS=[]
REGION_TILE_M=2000
qa_rng=np.random.default_rng(SEED+50)

with rasterio.open(DEM_FILES[2015]) as d15, \
     rasterio.open(DEM_FILES[2022]) as d22, \
     rasterio.open(DEM_FILES[2024]) as d24:
    for i,(r,c,h,w) in enumerate(TILES):
        win=Window(c,r,w,h)
        a15=read_core(d15,win)
        a22=read_core(d22,win)
        a24=read_core(d24,win)
        input22=read_with_halo(d22,win)
        feat22=build_terrain_features(input22,win)
        base=(np.isfinite(a15)&np.isfinite(a22)&np.isfinite(a24)&
              valid_feature_mask(input22,win)&
              (feat22[:,:,1]<5.0)&(feat22[:,:,14]<2.0))
        # Randomly sample candidate pixels to avoid keeping all ~1m pixels.
        candidate=np.flatnonzero(base.ravel())
        if len(candidate)>1500:
            candidate=qa_rng.choice(candidate,size=1500,replace=False)
        if not len(candidate):continue
        rr=candidate//w + r
        cc=candidate%w + c
        d1=(a22-a15).ravel()[candidate]
        d2=(a24-a22).ravel()[candidate]
        for a,b,values in [(2015,2022,d1),(2022,2024,d2)]:
            for k in np.unique((rr//REGION_TILE_M)*100000+(cc//REGION_TILE_M)):
                chosen=(rr//REGION_TILE_M)*100000+(cc//REGION_TILE_M)==k
                STABLE_QA_ROWS.append((f'{a}-{b}',int(k),values[chosen].astype('float32')))

from collections import defaultdict
by_pair=defaultdict(list)
by_block=defaultdict(list)
for period,block,values in STABLE_QA_ROWS:
    by_pair[period].append(values)
    by_block[(period,block)].append(values)
STABLE_QA={}
for period,ls in by_pair.items():
    vals=np.concatenate(ls)
    vals=vals[np.isfinite(vals)&(np.abs(vals)<4.0)]
    if len(vals)<100:continue
    med=float(np.median(vals));mad=float(1.4826*np.median(np.abs(vals-med)))
    regional=[]
    for (p,block),arrs in by_block.items():
        if p!=period:continue
        vv=np.concatenate(arrs)
        vv=vv[np.isfinite(vv)&(np.abs(vv)<4.0)]
        if len(vv)>=50:regional.append(float(np.median(vv)))
    region_range=(float(np.percentile(regional,95)-np.percentile(regional,5))
                  if len(regional)>=4 else None)
    STABLE_QA[period]={'candidate_points':len(vals),'median_m':med,
       'robust_spread_m':mad,'regional_p95_minus_p05_m':region_range,
       'regional_blocks':len(regional),'status':'candidate_terrain_only_no_geoid_correction'}
print(json.dumps(STABLE_QA,indent=2))
(OUT/'stable_ground_1m_diagnostic.json').write_text(json.dumps(STABLE_QA,indent=2))
print('No elevation correction applied. Independently stable reference surfaces are still required.')

In [ ]:
# CELL 10 — Predict 2024 relative susceptibility in 512-pixel tiles.
# Saves a CROPPED, georeferenced, 1 m float32 risk GeoTIFF, never a full RAM array.
from rasterio.windows import transform as win_transform
RISK_FILE=OUT/'ecosignal_georisk_v2_2024_risk_1m.tif'
output_window=Window(MIN_C,MIN_R,MAX_C-MIN_C,MAX_R-MIN_R)
output_transform=win_transform(output_window,GRID_TRANSFORM)

profile={
    'driver':'GTiff','width':int(output_window.width),
    'height':int(output_window.height),'count':1,'dtype':'float32',
    'crs':GRID_CRS,'transform':output_transform,
    'nodata':-9999.,'tiled':True,'blockxsize':512,'blockysize':512,
    'compress':'DEFLATE','predictor':3,'BIGTIFF':'IF_SAFER',
    'SPARSE_OK':'YES'
}
valid_predicted=0
with rasterio.open(DEM_FILES[2015]) as d15, \
     rasterio.open(DEM_FILES[2022]) as d22, \
     rasterio.open(DEM_FILES[2024]) as d24, \
     rasterio.open(RISK_FILE,'w',**profile) as dst:
    for i,(r,c,h,w) in enumerate(TILES):
        win=Window(c,r,w,h)
        z24=read_core(d24,win)
        footprint=(np.isfinite(read_core(d15,win))&
                   np.isfinite(read_core(d22,win))&np.isfinite(z24))
        source24=read_with_halo(d24,win)
        valid=footprint&valid_feature_mask(source24,win)
        if not valid.any():continue
        feat=build_terrain_features(source24,win)
        valid &= np.isfinite(feat).all(axis=-1)
        tileout=np.full((h,w),-9999.,dtype=np.float32)
        indices=np.flatnonzero(valid.ravel())
        view=feat.reshape(-1,len(FEATURE_NAMES))
        for start in range(0,len(indices),40000):
            inds=indices[start:start+40000]
            tileout.ravel()[inds]=FINAL_MODEL.predict_proba(view[inds])[:,1].astype('float32')
        relative_win=Window(c-MIN_C,r-MIN_R,w,h)
        dst.write(tileout,1,window=relative_win)
        valid_predicted+=len(indices)
        if (i+1)%25==0 or i+1==len(TILES):
            print(f'2024 prediction: tile {i+1}/{len(TILES)}')
    dst.update_tags(
        name='EcoSignal GeoRisk V2 terrain-lowering susceptibility',
        model=WINNER,grid_resolution_m='1',
        elevation_units='meters',vertical_reference='provisional',
        calibration='uncalibrated_relative_score',
        interpretation='Higher score means stronger relative propensity of observed terrain lowering',
        source_year='2024',training_labels='2015-2022',
        holdout_labels='2022-2024')
print('Saved:',RISK_FILE,'bytes:',RISK_FILE.stat().st_size)
print('Predicted 1 m pixels:',valid_predicted)
assert valid_predicted>1000,'Risk map is almost empty; inspect source validity.'

In [ ]:
# CELL 11 — Export lightweight PNG + WGS84 bounds for EcoSignal web overlay.
# This is for map display ONLY; use the 1m GeoTIFF for geospatial analysis.
from rasterio.warp import transform_bounds, reproject, Resampling
from rasterio.transform import from_bounds
from PIL import Image
import matplotlib.pyplot as plt

PNG_FILE=OUT/'ecosignal_georisk_v2_overlay.png'
OVERLAY_JSON=OUT/'ecosignal_georisk_v2_overlay.json'
PREVIEW_MAX_WIDTH=1300

with rasterio.open(RISK_FILE) as src:
    west,south,east,north=transform_bounds(src.crs,'EPSG:4326',*src.bounds,densify_pts=41)
    width=PREVIEW_MAX_WIDTH
    height=max(1,min(2200,int(round(width*(north-south)/max(east-west,1e-9)))))
    dst_transform=from_bounds(west,south,east,north,width,height)
    warped=np.full((height,width),np.nan,dtype=np.float32)
    reproject(source=rasterio.band(src,1),destination=warped,
              src_transform=src.transform,src_crs=src.crs,src_nodata=src.nodata,
              dst_transform=dst_transform,dst_crs='EPSG:4326',dst_nodata=np.nan,
              resampling=Resampling.average,num_threads=2)

valid=np.isfinite(warped)&(warped>=0)&(warped<=1)
rgba=(plt.get_cmap('inferno')(np.clip(np.nan_to_num(warped,nan=0),0,1))*255).astype(np.uint8)
rgba[:,:,3]=np.where(valid,205,0).astype(np.uint8)
Image.fromarray(rgba,'RGBA').save(PNG_FILE,optimize=True)

info={
  'image_filename':PNG_FILE.name,
  'image_dimensions_px':[width,height],
  'bounds_wgs84':{'west':west,'south':south,'east':east,'north':north},
  'leaflet_image_overlay_bounds':[[south,west],[north,east]],
  'layer_name':'GeoRisk V2 — terrain-lowering susceptibility',
  'color_map':'inferno','opacity_embedded_in_png':True,
  'score_scale':[0.0,1.0],
  'interpretation':'Relative model score; not a calibrated erosion probability',
  'data_year':2024,
  'limitations':['Vertical datum/geoid equivalence not independently confirmed',
                 'Provisional 2022–2024 label uncertainty',
                 'Terrain-only features: no global hydrology/catchment flow',
                 'Raster may include natural, construction or LiDAR processing changes',
                 'PNG downsampled for browser; authoritative raster is 1 m GeoTIFF']
}
OVERLAY_JSON.write_text(json.dumps(info,indent=2))
print('PNG:',PNG_FILE,'KB:',round(PNG_FILE.stat().st_size/1024))
print('Geographic overlay bounds:',info['leaflet_image_overlay_bounds'])
print('Manifest:',OVERLAY_JSON)

In [ ]:
# CELL 12 — Export reproducible model, scores, geospatial metadata, and app integration guide.
import sklearn, xgboost, json, datetime
MODEL_FILE=OUT/'ecosignal_georisk_v2_1m.joblib'
METADATA_FILE=OUT/'ecosignal_georisk_v2_1m_metadata.json'
RESULTS_CSV=OUT/'model_selection_2015_2022_dev.csv'
GUIDE_FILE=OUT/'ECOSIGNAL_INTEGRATION_README.md'

joblib.dump(FINAL_MODEL,MODEL_FILE,compress=3)
pd.DataFrame.from_dict(DEV_RESULTS,orient='index').to_csv(RESULTS_CSV)
meta={
 'name':'EcoSignal GeoRisk V2, 1 m windowed terrain model',
 'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'model_file':MODEL_FILE.name,
 'model_type':WINNER,'feature_order':FEATURE_NAMES,
 'frozen_classification_threshold':FROZEN_THRESHOLD,
 'feature_implementation_version':FEATURE_IMPLEMENTATION_VERSION,
 'frozen_model_checkpoint':FROZEN_MODEL_FILE.name,
 'holdout_checkpoint':EVALUATION_FILE.name,
 'feature_definition':'Cell 3 of companion notebook',
 'training_input_year':2015,'training_label_period':'2015-2022',
 'test_input_year':2022,'test_label_period':'2022-2024',
 'latest_map_input_year':2024,
 'prediction_semantics':'relative uncalibrated susceptibility to observed terrain lowering',
 'uncertainty_thresholds_m':{
  '2015_2022':LOWERING_THRESHOLD_2015_2022_M,
  '2022_2024':LOWERING_THRESHOLD_2022_2024_M,
  'ambiguity_band':LABEL_AMBIGUITY_BAND_M},
 'source_grid':{'crs':str(GRID_CRS),'resolution_m':1,'shape':[H,W]},
 'horizontal_grid':'aligned original USGS 2015 EPSG:32119 grid',
 'vertical_units':'meters',
 'vertical_reference_status':'PROVISIONAL',
 'geoid_assumptions':{
    '2013':'GEOID12A assumed; not used for model',
    '2015':'GEOID12A source documented',
    '2022':'GEOID18 assumed; not independently verified',
    '2024':'GEOID18 documented'},
 'vertical_offset_applied_m':0.0,
 'spatial_validation':{
    'block_m':BLOCK_M,'boundary_buffer_m':BUFFER_M,
    'train_percent_of_block_hash':65,'dev_percent':20,'final_test_percent':15},
 'training_sample_by_class_upper_bound':TRAIN_SAMPLES_PER_CLASS,
 'development_max_samples':DEV_SAMPLES,'temporal_test_max_samples':TEST_SAMPLES,
 'development_2015_2022':DEV_RESULTS,
 'temporal_spatial_holdout_2022_2024':FINAL_METRICS,
 'candidate_stable_ground_diagnostic_1m':STABLE_QA if 'STABLE_QA' in globals() else {},
 'risk_raster':RISK_FILE.name,'web_overlay_png':PNG_FILE.name,
 'web_overlay_manifest':OVERLAY_JSON.name,
 'pre2024_sample_cache_signature':CACHE_SIGNATURE,
 'libraries':{'rasterio':rasterio.__version__, 'sklearn':sklearn.__version__,
              'xgboost':xgboost.__version__},
 'critical_limitations':[
    '2015-2022 (7 yr) and 2022-2024 (2 yr) are different observation horizons',
    'Source-specific combined 2022-2024 vertical uncertainty not verified',
    'Geoid reference assumptions do not establish survey datum consistency',
    'Negative and positive classes exclude a narrow ambiguous lowering band',
    'Pixel samples spatially autocorrelated even when evaluation blocks held out',
    'Model does not estimate exact amount of future erosion, absolute risk or erosion volume',
    'Construction, water, bare-earth generation and change detection errors may mimic erosion',
    'Tile-safe predictors exclude global hydrological accumulation and TWI',
    'Independent field/ground-control confirmation not performed']
}
METADATA_FILE.write_text(json.dumps(meta,indent=2,default=str))
GUIDE_FILE.write_text(f"""# EcoSignal GeoRisk V2 integration

Files:
- `{MODEL_FILE.name}` — trained model; depends on notebook Cell 3 for exact features.
- `{RISK_FILE.name}` — authoritative 1 m EPSG:32119 float GeoTIFF. NoData = -9999.
- `{PNG_FILE.name}` — downsampled transparent WGS84 web overlay.
- `{OVERLAY_JSON.name}` — geographic bounds and legend metadata.
- `{METADATA_FILE.name}` — model metrics, feature definitions, limitations.

For Leaflet, host the PNG as a static asset, load `leaflet_image_overlay_bounds` from the JSON, and create `L.imageOverlay(pngUrl, bounds, {{opacity:1}})`. The PNG already contains alpha; adjust the map layer's opacity separately. For MapLibre use an `image` source with WGS84 corner coordinates. The overlay is only a visualization, not the native raster.

**Suggested UI wording:** GeoRisk V2 — relative terrain-lowering susceptibility (2024 LiDAR baseline). Survey/geoid compatibility and 2022–2024 vertical uncertainty are provisional. Not a forecast of exact erosion depth or a calibrated probability.

**Do not alter** the existing satellite, AQI, coverage, growth-form, geomorphology and Ask LichenLens features while adding this layer. There is no automatic website deployment in this notebook.
""")
print('All exports saved to:',OUT)
for path in [MODEL_FILE,METADATA_FILE,RESULTS_CSV,RISK_FILE,PNG_FILE,OVERLAY_JSON,GUIDE_FILE]:
    print('  ',path.name,'—',round(path.stat().st_size/1e6,3),'MB')
print('The app has NOT been modified or published. Use these artifacts for integration.')

## Final run checklist

1. **Before Cell 1:** complete 1 m alignment in `aligned_dems_1m` (all outputs fully written). Keep old originals and the saved V1 model untouched.
2. **Cells 1–4:** dependencies, paths/grid check, functions, scan common coverage. A warning or error indicates a mismatch, not a license to bypass scientific QA.
3. **Cells 5–7:** build **pre-2024** training and geographic development datasets. The 2024 survey is used only as a coverage mask; 2022–2024 changes are not examined until the model is frozen.
4. **Cell 8:** record the **first and only independent temporal+spatial test**. Do not tune against it afterward and still report it as independent.
5. **Cell 9:** examine provisional **1 m candidate stable-ground** offsets without modifying any DEM. This is not independently validated datum coregistration.
6. **Cells 10–12:** create cropped 1 m risk GeoTIFF, browser PNG overlay, model, metrics, metadata and integration instructions. They are saved to Drive.

**How to interpret the result:** model outputs are **relative scores** based on local 2024 terrain and historical 2015–2022 change-detection labels. Results cannot certify that terrain lowering is erosion or estimate a 2024–2026 erosion rate. For a validated physical erosion model, obtain independent ground truth, QA construction/water masks, confirm 2022/2024 vertical compatibility, and generate catchment-aware hydrologic predictors.

**Memory caveat:** Raster windows are bounded, but processing the full 1 m overlap may still take a long time and consume significant Drive I/O. The code saves a reproducible pre-2024 sample cache. If full overlap is extremely large, choose a smaller independent study AOI *before* training, not after looking at final holdout results.